[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msfasha/307401-Big-Data/blob/main/20252/Module%205%20-%20Introduction%20to%20Spark/2-RDDs_Transformations_and_Actions.ipynb)

<h1 style="color:#E25822; font-family:Arial; font-size:2.2em;">Introduction to PySpark: RDDs, Transformations &amp; Actions</h1>

**Course:** Big Data Analytics &nbsp;|&nbsp; **Module 5:** Introduction to Apache Spark

---

## Learning Objectives

By the end of this notebook you will be able to:

- Explain what an RDD is and describe its three core properties
- Create RDDs from Python collections, text files, and ranges
- Apply common **transformations** (lazy operations) to RDDs
- Trigger computation using **actions** (eager operations)
- Understand Spark's **lazy evaluation** model and why it matters
- Work with **key-value pair RDDs** for aggregation tasks
- Build a complete **Word Count** pipeline from scratch

---

### Notebook Map

| # | Section | Key Concept |
|---|---------|-------------|
| 1 | Setup | Install PySpark, create SparkSession |
| 2 | RDD Introduction | What is an RDD? |
| 3 | Creating RDDs | `parallelize`, `textFile`, `range` |
| 4 | Transformations | `map`, `filter`, `flatMap`, `union`, ... |
| 5 | Actions | `collect`, `count`, `reduce`, ... |
| 6 | Lazy Evaluation | DAG, benefits |
| 7 | Word Count Example | End-to-end pipeline |
| 8 | Key-Value RDDs | Pair RDDs, `groupByKey`, `reduceByKey` |
| 9 | Exercises | Practice problems |

---
<h2 style="color:#E25822; font-family:Arial;">1 — Environment Setup</h2>

Google Colab does not ship with PySpark pre-installed, so we install it first.  
We then create a **`SparkSession`** — the single entry point to all Spark functionality — and inspect how many CPU cores are available to our local cluster.

In [ ]:
# Install PySpark (only needed in Colab / fresh environments)
!pip install pyspark --quiet

In [ ]:
import multiprocessing
from pyspark.sql import SparkSession

# ---------------------------------------------------------------------------
# How many CPU cores does this machine expose?
# ---------------------------------------------------------------------------
cpu_cores = multiprocessing.cpu_count()
print(f"Available CPU cores : {cpu_cores}")

# ---------------------------------------------------------------------------
# Create a SparkSession
#   - appName  : human-readable name shown in the Spark UI
#   - master   : 'local[*]' uses ALL available cores on this machine
# ---------------------------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("RDDs_Transformations_Actions")
    .master(f"local[{cpu_cores}]")
    .getOrCreate()
)

# SparkContext is the lower-level API for RDD operations
sc = spark.sparkContext
sc.setLogLevel("ERROR")  # suppress INFO/WARN noise

print(f"Spark version       : {spark.version}")
print(f"SparkContext master : {sc.master}")
print("SparkSession ready!")

---
<h2 style="color:#E25822; font-family:Arial;">2 — What is an RDD?</h2>

An **RDD (Resilient Distributed Dataset)** is the fundamental data abstraction in Apache Spark.  
It represents an **immutable**, **partitioned** collection of records that can be processed in parallel across a cluster.

### The Three Core Properties

| Property | Meaning |
|----------|---------|
| **Resilient** | Fault-tolerant — if a partition is lost, Spark can recompute it from the **lineage** (the sequence of transformations that created it) |
| **Distributed** | Data is split into **partitions** spread across worker nodes, enabling true parallel processing |
| **Dataset** | A collection of records — can be structured (rows/columns) or unstructured (raw text, bytes) |

```
  Driver Program
       │
       │  SparkContext
       │
  ┌────▼──────────────────────────────────────────┐
  │              RDD  (logical view)               │
  │  ┌──────────┐  ┌──────────┐  ┌──────────┐    │
  │  │Partition 0│  │Partition 1│  │Partition 2│   │
  │  │ Worker 1  │  │ Worker 2  │  │ Worker 3  │   │
  │  └──────────┘  └──────────┘  └──────────┘    │
  └───────────────────────────────────────────────┘
```

### When to Use RDDs

- You need **fine-grained control** over how data is partitioned or processed
- You are working with **unstructured data** (log files, raw text, binary)
- You need **functional programming** primitives (map, filter, reduce)
- You are using Spark **MLlib** or **GraphX** APIs that still expose RDDs
- The DataFrame / Dataset API does not cover your use case

> **Note:** For most structured-data workflows, prefer DataFrames — they benefit from Spark's Catalyst optimizer and Tungsten execution engine. RDDs are the foundation, but DataFrames build on top of them.

---
<h2 style="color:#E25822; font-family:Arial;">3 — Creating RDDs</h2>

There are three common ways to create an RDD:

1. **`sc.parallelize()`** — distribute an existing Python collection
2. **`sc.textFile()`** — read lines from a text file
3. **`sc.range()`** — generate a range of integers

Each method accepts an optional `numSlices` (or `minPartitions`) argument to control the number of partitions.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Method 1: sc.parallelize() — from a Python list
# ─────────────────────────────────────────────────────────────
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
numbers_rdd = sc.parallelize(numbers, numSlices=4)  # split into 4 partitions

print("Type            :", type(numbers_rdd))
print("Number of items :", numbers_rdd.count())
print("Partitions      :", numbers_rdd.getNumPartitions())
print("Contents        :", numbers_rdd.collect())

In [ ]:
# ─────────────────────────────────────────────────────────────
# Method 2: sc.textFile() — from a text file
#   Each *line* in the file becomes one element in the RDD.
#   We write a small temp file first so this cell is self-contained.
# ─────────────────────────────────────────────────────────────
import tempfile, os

# Write a small sample file
sample_text = """Apache Spark is a fast and general-purpose cluster computing system.
It provides high-level APIs in Java, Scala, Python and R.
Spark also supports a rich set of higher-level tools.
These include Spark SQL, MLlib, GraphX, and Spark Streaming."""

tmp_file = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False)
tmp_file.write(sample_text)
tmp_file.close()

# Load the file as an RDD of lines
text_rdd = sc.textFile(tmp_file.name)

print("Lines in file:")
for line in text_rdd.collect():
    print(" ", line)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Method 3: sc.range() — integer range (like Python's range())
#   Useful for quick testing and benchmarking.
# ─────────────────────────────────────────────────────────────
range_rdd = sc.range(0, 20, step=2)  # even numbers 0, 2, 4, ..., 18

print("Even numbers from 0 to 18:")
print(range_rdd.collect())
print("Count:", range_rdd.count())

---
<h2 style="color:#E25822; font-family:Arial;">4 — Transformations (Lazy Operations)</h2>

A **transformation** takes one RDD and returns a **new RDD** — it does **not** execute immediately.  
Spark builds a **DAG (Directed Acyclic Graph)** of all transformations and only runs them when an **action** is called.

This approach is called **lazy evaluation** and is covered in depth in Section 6.

Common transformations:

| Transformation | Description |
|----------------|-------------|
| `map(f)` | Apply `f` to each element, returning one output per input |
| `filter(f)` | Keep only elements where `f` returns `True` |
| `flatMap(f)` | Like `map`, but `f` returns an iterable — results are flattened |
| `distinct()` | Remove duplicate elements |
| `union(other)` | Concatenate two RDDs |
| `sortBy(f)` | Sort elements using a key function |
| `groupByKey()` | Group values by key (for pair RDDs) |
| `reduceByKey(f)` | Aggregate values by key using `f` (more efficient than `groupByKey`) |

In [ ]:
# ─────────────────────────────────────────────────────────────
# map() — apply a function to every element
#   Here: square each number in the RDD
# ─────────────────────────────────────────────────────────────
nums = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# Using a lambda function
squared_rdd = nums.map(lambda x: x ** 2)

print("Original :", nums.collect())
print("Squared  :", squared_rdd.collect())

In [ ]:
# ─────────────────────────────────────────────────────────────
# filter() — keep only elements that satisfy a condition
#   Here: keep even numbers
# ─────────────────────────────────────────────────────────────
evens_rdd = nums.filter(lambda x: x % 2 == 0)

print("All numbers :", nums.collect())
print("Even numbers:", evens_rdd.collect())

In [ ]:
# ─────────────────────────────────────────────────────────────
# flatMap() — map + flatten
#   Here: split sentences into individual words.
#   map() would produce a list of lists; flatMap() flattens to one list.
# ─────────────────────────────────────────────────────────────
sentences = sc.parallelize([
    "Spark is fast",
    "Spark is scalable",
    "Big Data is fun"
])

# map()     → [['Spark','is','fast'], ['Spark','is','scalable'], ...]
# flatMap() → ['Spark','is','fast','Spark','is','scalable', ...]
words_rdd = sentences.flatMap(lambda line: line.split(" "))

print("Using map()    :", sentences.map(lambda l: l.split()).collect())
print("Using flatMap():", words_rdd.collect())

In [ ]:
# ─────────────────────────────────────────────────────────────
# distinct() — remove duplicate elements
# ─────────────────────────────────────────────────────────────
dupes = sc.parallelize([1, 2, 2, 3, 3, 3, 4, 5, 5])
unique_rdd = dupes.distinct()

print("With duplicates :", sorted(dupes.collect()))
print("After distinct() :", sorted(unique_rdd.collect()))

In [ ]:
# ─────────────────────────────────────────────────────────────
# union() — combine two RDDs into one (no deduplication)
# ─────────────────────────────────────────────────────────────
rdd_a = sc.parallelize(["apple", "banana", "cherry"])
rdd_b = sc.parallelize(["cherry", "date", "elderberry"])

combined_rdd = rdd_a.union(rdd_b)

print("RDD A     :", rdd_a.collect())
print("RDD B     :", rdd_b.collect())
print("union()   :", combined_rdd.collect())  # 'cherry' appears twice

In [ ]:
# ─────────────────────────────────────────────────────────────
# sortBy() — sort elements by a key function
#   ascending=False → descending order
# ─────────────────────────────────────────────────────────────
unsorted = sc.parallelize([5, 3, 8, 1, 9, 2, 7, 4, 6])

asc_rdd  = unsorted.sortBy(lambda x: x)          # ascending
desc_rdd = unsorted.sortBy(lambda x: x, ascending=False)  # descending

print("Original   :", unsorted.collect())
print("Ascending  :", asc_rdd.collect())
print("Descending :", desc_rdd.collect())

In [ ]:
# ─────────────────────────────────────────────────────────────
# groupByKey() vs reduceByKey() — key-value pair RDDs
#
# groupByKey  : groups ALL values for each key → sends data over network
# reduceByKey : combines values locally first → much more efficient
# ─────────────────────────────────────────────────────────────
pairs = sc.parallelize([
    ("math", 90), ("science", 85), ("math", 78),
    ("science", 92), ("math", 88), ("english", 76)
])

# groupByKey — collect all values per key
grouped = pairs.groupByKey().mapValues(list)
print("groupByKey():")
for subject, scores in sorted(grouped.collect()):
    print(f"  {subject}: {scores}")

print()

# reduceByKey — sum scores per subject (more efficient)
totals = pairs.reduceByKey(lambda a, b: a + b)
print("reduceByKey (sum):")
for subject, total in sorted(totals.collect()):
    print(f"  {subject}: {total}")

## The real difference is data movement (shuffle)

### groupByKey()
- Sends **ALL values** across the network  
- Then groups them  
- High memory usage  
- High network cost  

### reduceByKey()
- Combines values **locally on each partition first**  
- Sends only **partial results**  
- Much less data shuffled  

---
<h2 style="color:#E25822; font-family:Arial;">5 — Actions (Eager Operations)</h2>

An **action** triggers the execution of all queued transformations and returns a result to the driver program (or writes data to storage).

Common actions:

| Action | Returns | Description |
|--------|---------|-------------|
| `collect()` | Python list | Fetch all elements to the driver — use only on small RDDs! |
| `count()` | int | Number of elements |
| `first()` | single element | First element |
| `take(n)` | Python list | First `n` elements |
| `reduce(f)` | single value | Aggregate all elements with a binary function |
| `countByValue()` | dict | Frequency of each distinct element |
| `saveAsTextFile(path)` | None | Write each element as a line in a text file |

> **Warning:** `collect()` pulls all data to the driver. On a large RDD this can cause an out-of-memory error. Prefer `take(n)` for inspection.

In [ ]:
# ─────────────────────────────────────────────────────────────
# collect(), count(), first(), take(n)
# ─────────────────────────────────────────────────────────────
data = sc.parallelize(range(1, 21))  # 1 to 20

print("collect()   :", data.collect())
print("count()     :", data.count())
print("first()     :", data.first())
print("take(5)     :", data.take(5))
print("takeSample  :", data.takeSample(withReplacement=False, num=5, seed=42))

In [ ]:
# ─────────────────────────────────────────────────────────────
# reduce() — aggregate all elements into a single value
#   The function must be commutative and associative.
#   Here: compute the sum of 1..20
# ─────────────────────────────────────────────────────────────
total = data.reduce(lambda a, b: a + b)
print(f"Sum of 1 to 20 = {total}")
print(f"Verify (Python sum): {sum(range(1, 21))}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# countByValue() — frequency count for each unique element
# ─────────────────────────────────────────────────────────────
colours = sc.parallelize(["red", "blue", "red", "green", "blue", "red", "yellow"])

freq = colours.countByValue()   # returns a Python defaultdict
print("Colour frequencies:")
for colour, count in sorted(freq.items()):
    print(f"  {colour}: {count}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# saveAsTextFile() — write RDD to disk
#   Spark creates a directory and writes one file per partition.
# ─────────────────────────────────────────────────────────────
import tempfile, shutil

output_dir = tempfile.mkdtemp() + "/rdd_output"

small_rdd = sc.parallelize(["Line one", "Line two", "Line three"], numSlices=2)
small_rdd.saveAsTextFile(output_dir)

print(f"Output directory: {output_dir}")
print("Files created:")
for fname in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, fname)
    if fname.startswith("part"):
        with open(fpath) as f:
            content = f.read().strip()
        print(f"  {fname}: '{content}'")

---
<h2 style="color:#E25822; font-family:Arial;">6 — Lazy Evaluation Deep Dive</h2>

### How it works

When you call a **transformation**, Spark does **nothing** — it only records the operation in the lineage graph (DAG).  
When you call an **action**, Spark compiles the entire DAG into an execution plan and runs it.

```
  sc.parallelize([1..10])
         │
    [Transformation]  .map(x → x²)          ← recorded, NOT executed
         │
    [Transformation]  .filter(x > 25)        ← recorded, NOT executed
         │
    [Transformation]  .sortBy(x)             ← recorded, NOT executed
         │
    [ACTION]          .collect()             ← TRIGGERS execution of all above
         │
    Result returned to driver
```

### Why Lazy Evaluation?

| Benefit | Explanation |
|---------|-------------|
| **Query optimization** | Spark can reorder, combine, or eliminate steps before running — similar to a SQL query planner |
| **Avoid unnecessary work** | If you call `.first()` on a filtered RDD, Spark stops as soon as one element is found |
| **Pipeline fusion** | Multiple transformations can be chained into a single pass over the data, reducing I/O |
| **Fault tolerance** | The lineage DAG is stored — if a partition fails, Spark re-runs only the steps needed to rebuild it |

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demonstrating lazy evaluation:
#   - Transformations return immediately (no computation)
#   - toDebugString() shows the DAG lineage
#   - Only .collect() triggers real work
# ─────────────────────────────────────────────────────────────
import time

base = sc.parallelize(range(1, 1_000_001))  # 1 million numbers

# These transformations return INSTANTLY — no data is processed yet
t0 = time.time()
step1 = base.map(lambda x: x * 2)         # double each element
step2 = step1.filter(lambda x: x % 6 == 0) # keep multiples of 6
step3 = step2.map(lambda x: x // 3)        # divide by 3
t1 = time.time()
print(f"3 transformations defined in {t1 - t0:.4f} seconds (lazy — no work done yet)")

# Print the DAG lineage
print("\nLineage (DAG):")
print(step3.toDebugString().decode())

# NOW collect() triggers execution
t2 = time.time()
result = step3.take(10)
t3 = time.time()
print(f"\nAction .take(10) executed in {t3 - t2:.4f} seconds")
print("First 10 results:", result)

---
<h2 style="color:#E25822; font-family:Arial;">Transformations vs Actions — Summary Table</h2>

| Feature | Transformation | Action |
|---------|---------------|--------|
| **Execution** | Lazy — deferred | Eager — immediate |
| **Returns** | New RDD | Value / side-effect |
| **Examples** | `map`, `filter`, `flatMap`, `distinct`, `union`, `sortBy`, `groupByKey`, `reduceByKey` | `collect`, `count`, `first`, `take`, `reduce`, `countByValue`, `saveAsTextFile` |
| **Triggers a Spark job?** | No | Yes |
| **Can be chained?** | Yes — builds the DAG | No — ends the chain |
| **Stored in lineage?** | Yes | No |

---
<h2 style="color:#E25822; font-family:Arial;">7 — Practical Example: Word Count Pipeline</h2>

**Word Count** is the "Hello World" of distributed computing.  
Given a body of text, count how many times each word appears.

### Pipeline Steps

```
Raw Text String
      │
  sc.parallelize()          → RDD of lines
      │
  .flatMap(split)           → RDD of words  (one word per element)
      │
  .map(word → (word, 1))    → RDD of (word, count) pairs
      │
  .reduceByKey(+)           → RDD of (word, total_count)
      │
  .sortBy(count, desc)      → sorted from most to least frequent
      │
  .take(10)                 → top 10 words
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# Step 0: Sample text (no external file needed)
# ─────────────────────────────────────────────────────────────
sample_corpus = """
Apache Spark is an open source distributed computing framework.
Spark provides an interface for programming entire clusters with
implicit data parallelism and fault tolerance. Originally developed
at UC Berkeley in 2009, Spark has grown to be one of the most
popular big data processing engines in the world. Spark can be
used for batch processing, stream processing, machine learning,
and graph processing. Spark supports multiple programming languages
including Python, Java, Scala, and R. PySpark is the Python API
for Apache Spark, enabling Python developers to harness the power
of distributed computing. Spark RDDs provide a low level API
for distributed data processing. Spark also offers higher level
APIs such as DataFrames and Datasets. Big data processing with
Spark is fast because Spark keeps data in memory between operations.
"""

print("Sample corpus loaded.")
print(f"Total characters: {len(sample_corpus)}")

In [ ]:
import re

# ─────────────────────────────────────────────────────────────
# Step 1: Split corpus into lines and load into an RDD
# ─────────────────────────────────────────────────────────────
lines_rdd = sc.parallelize(sample_corpus.strip().split("\n"))
print(f"Step 1 — Lines : {lines_rdd.count()}")
print("  First line  :", lines_rdd.first())

In [ ]:
# ─────────────────────────────────────────────────────────────
# Step 2: flatMap — split each line into words
#   Also: lowercase + strip punctuation using regex
# ─────────────────────────────────────────────────────────────
def clean_and_split(line):
    # Remove non-alpha characters and split on whitespace
    words = re.sub(r"[^a-zA-Z\s]", "", line).lower().split()
    return words

words_rdd = lines_rdd.flatMap(clean_and_split)
print(f"Step 2 — Total words : {words_rdd.count()}")
print("  Sample words :", words_rdd.take(10))

In [ ]:
# ─────────────────────────────────────────────────────────────
# Step 3: map — convert each word to a (word, 1) pair
# ─────────────────────────────────────────────────────────────
pairs_rdd = words_rdd.map(lambda word: (word, 1))
print("Step 3 — Word pairs (first 5):")
for pair in pairs_rdd.take(5):
    print(" ", pair)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Step 4: reduceByKey — sum the 1s for each word
# ─────────────────────────────────────────────────────────────
word_counts_rdd = pairs_rdd.reduceByKey(lambda a, b: a + b)
print(f"Step 4 — Distinct words : {word_counts_rdd.count()}")
print("  Sample counts:", word_counts_rdd.take(5))

In [ ]:
# ─────────────────────────────────────────────────────────────
# Step 5: sortBy — rank words by frequency (descending)
# ─────────────────────────────────────────────────────────────
ranked_rdd = word_counts_rdd.sortBy(lambda pair: pair[1], ascending=False)

# Step 6: take(10) — retrieve top 10 words
top_10 = ranked_rdd.take(10)

print("Top 10 most frequent words:")
print(f"  {'Rank':<5} {'Word':<15} {'Count':<5}")
print("  " + "-" * 27)
for rank, (word, count) in enumerate(top_10, start=1):
    print(f"  {rank:<5} {word:<15} {count:<5}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# One-liner: the full pipeline chained together
# ─────────────────────────────────────────────────────────────
top_words = (
    sc.parallelize(sample_corpus.strip().split("\n"))  # RDD of lines
      .flatMap(clean_and_split)                         # flatten to words
      .map(lambda w: (w, 1))                            # (word, 1) pairs
      .reduceByKey(lambda a, b: a + b)                  # aggregate counts
      .sortBy(lambda p: p[1], ascending=False)          # sort by frequency
      .take(10)                                         # top 10
)

print("Chained pipeline result (top 10):")
for word, count in top_words:
    bar = "█" * count
    print(f"  {word:<15} {count:>3}  {bar}")

---
<h2 style="color:#E25822; font-family:Arial;">8 — Key-Value Pair RDDs</h2>

A **Pair RDD** is an RDD where each element is a **2-tuple `(key, value)`**.  
Pair RDDs unlock a rich set of key-aware operations:

| Operation | Description |
|-----------|-------------|
| `groupByKey()` | Group all values for each key into an iterable |
| `reduceByKey(f)` | Merge values for each key using a binary function |
| `mapValues(f)` | Apply `f` to each value without changing the key |
| `keys()` | Return an RDD of just the keys |
| `values()` | Return an RDD of just the values |
| `sortByKey()` | Sort by key |
| `join(other)` | Inner join two pair RDDs on their keys |
| `countByKey()` | Count how many values each key has |

**Example scenario:** Sales transactions by region.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Create a Pair RDD from a list of (region, sale_amount) tuples
# ─────────────────────────────────────────────────────────────
sales_data = [
    ("North",  1200), ("South",   850), ("East",  950),
    ("West",  1100),  ("North",   780), ("South", 1340),
    ("East",   620),  ("West",    900), ("North", 1050),
    ("South",  760),  ("East",   1400), ("West",   550),
]

sales_rdd = sc.parallelize(sales_data)

print("Sample sales records:")
for region, amount in sales_rdd.take(5):
    print(f"  Region: {region:<8} Sale: ${amount}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# keys() and values()
# ─────────────────────────────────────────────────────────────
regions    = sales_rdd.keys().distinct().collect()
all_sales  = sales_rdd.values().collect()

print("Distinct regions :", sorted(regions))
print("All sale amounts :", sorted(all_sales))

In [ ]:
# ─────────────────────────────────────────────────────────────
# reduceByKey — total sales per region
# ─────────────────────────────────────────────────────────────
total_by_region = sales_rdd.reduceByKey(lambda a, b: a + b)

print("Total sales by region:")
for region, total in sorted(total_by_region.collect()):
    print(f"  {region:<8}: ${total:,}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# mapValues — apply a function to values without touching keys
#   Here: apply a 10% bonus to every sale amount
# ─────────────────────────────────────────────────────────────
with_bonus = sales_rdd.mapValues(lambda amount: round(amount * 1.10, 2))

print("Sales with 10% bonus (first 5):")
for region, amount in with_bonus.take(5):
    print(f"  {region:<8}: ${amount}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# groupByKey — group all sales amounts per region
# ─────────────────────────────────────────────────────────────
grouped_sales = sales_rdd.groupByKey().mapValues(sorted)

print("Individual sales per region:")
for region, amounts in sorted(grouped_sales.collect()):
    print(f"  {region:<8}: {list(amounts)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# countByKey — how many transactions per region?
# ─────────────────────────────────────────────────────────────
tx_count = sales_rdd.countByKey()

print("Number of transactions per region:")
for region, count in sorted(tx_count.items()):
    print(f"  {region:<8}: {count} transactions")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Combining operations: average sale per region
#   Strategy: reduceByKey on (sum, count) tuples, then mapValues
# ─────────────────────────────────────────────────────────────
avg_by_region = (
    sales_rdd
    .mapValues(lambda v: (v, 1))                           # (region, (amount, 1))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) # sum amounts and counts
    .mapValues(lambda s_c: round(s_c[0] / s_c[1], 2))     # compute average
    .sortByKey()
)

print("Average sale amount per region:")
for region, avg in avg_by_region.collect():
    print(f"  {region:<8}: ${avg:,.2f}")

---
<h2 style="color:#E25822; font-family:Arial;">9 — Exercises</h2>

Work through the three exercises below. Each builds on the concepts introduced in this notebook.

> **Tip:** Use `rdd.collect()` to inspect intermediate results while building your solution. Switch to `.take(n)` once you are confident.

---

### Exercise 1 — Basic Transformations (Beginner)

Given the list of temperatures in Celsius below:

```python
temps_c = [0, 10, 20, 25, 30, 35, 40, 100, -5, 15]
```

1. Create an RDD from this list.
2. Convert every temperature to Fahrenheit using `map()`.  
   Formula: `F = C × 9/5 + 32`
3. Filter to keep only temperatures **above 80°F**.
4. Sort the result in **ascending** order.
5. Print the final result using `collect()`.

In [ ]:
# ── Exercise 1 — Your solution here ────────────────────────────

temps_c = [0, 10, 20, 25, 30, 35, 40, 100, -5, 15]

# TODO: Step 1 - Create RDD
# temps_rdd = ...

# TODO: Step 2 - Convert to Fahrenheit
# temps_f_rdd = ...

# TODO: Step 3 - Filter above 80°F
# hot_rdd = ...

# TODO: Step 4 - Sort ascending
# sorted_rdd = ...

# TODO: Step 5 - Collect and print
# print(sorted_rdd.collect())

---

### Exercise 2 — Key-Value Aggregation (Intermediate)

You are given a list of `(student_name, exam_score)` pairs:

```python
scores = [
    ("Alice", 85), ("Bob", 72), ("Alice", 90),
    ("Carol", 78), ("Bob", 88), ("Carol", 95),
    ("Alice", 76), ("Bob", 65), ("Carol", 82)
]
```

1. Create a Pair RDD from this list.
2. Compute the **total score** for each student using `reduceByKey()`.
3. Compute the **average score** for each student.  
   Hint: use `mapValues()` on a `(sum, count)` intermediate RDD.
4. Find the student with the **highest average** score.  
   Hint: `.sortBy()` or `reduce()`.
5. Print each student's name and average score, sorted alphabetically.

In [ ]:
# ── Exercise 2 — Your solution here ────────────────────────────

scores = [
    ("Alice", 85), ("Bob", 72), ("Alice", 90),
    ("Carol", 78), ("Bob", 88), ("Carol", 95),
    ("Alice", 76), ("Bob", 65), ("Carol", 82)
]

# TODO: Step 1 - Create Pair RDD

# TODO: Step 2 - Total score per student

# TODO: Step 3 - Average score per student

# TODO: Step 4 - Top student

# TODO: Step 5 - Print sorted by name

---

### Exercise 3 — Mini Text Analytics Pipeline (Advanced)

Using the `sample_corpus` string defined in the Word Count section:

1. Build a word-count RDD (reuse the pipeline from Section 7).
2. **Filter out** common English stop words:  
   `stop_words = {"the", "a", "an", "is", "are", "and", "or", "in", "of", "to", "for", "by", "as", "be", "has", "at", "its"}`
3. Keep only words with **length ≥ 4** characters.
4. Show the **top 10** words after filtering.
5. **Bonus:** Compute the total number of unique words remaining after all filters, and the total word count (sum of all frequencies).

In [ ]:
# ── Exercise 3 — Your solution here ────────────────────────────

stop_words = {
    "the", "a", "an", "is", "are", "and", "or",
    "in", "of", "to", "for", "by", "as", "be",
    "has", "at", "its"
}

# TODO: Step 1 - Word count pipeline (reuse from Section 7)

# TODO: Step 2 - Filter out stop words

# TODO: Step 3 - Keep only words with length >= 4

# TODO: Step 4 - Top 10 words

# TODO: Step 5 (Bonus) - Unique word count and total word frequency

---
<h2 style="color:#E25822; font-family:Arial;">Cleanup</h2>

Always stop your `SparkSession` when you are finished to release cluster resources.

In [ ]:
# Stop the SparkSession
spark.stop()
print("SparkSession stopped.")

---
<h2 style="color:#E25822; font-family:Arial;">Summary</h2>

In this notebook you learned:

- An **RDD** is the core abstraction in Spark — Resilient, Distributed, Dataset
- RDDs can be created from Python collections (`parallelize`), files (`textFile`), or ranges (`range`)
- **Transformations** are lazy — they build a DAG but do not execute until an action is called
- **Actions** trigger execution and return results to the driver or write to storage
- **Lazy evaluation** enables query optimization, pipeline fusion, and efficient fault recovery
- **Pair RDDs** enable key-aware aggregations: `groupByKey`, `reduceByKey`, `mapValues`, `join`
- The **Word Count** pipeline demonstrates how to chain transformations into a complete data-processing workflow

### Further Reading

- [Apache Spark RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
- [PySpark API Reference — RDD](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html)
- [Learning Spark, 2nd Edition — Chapter 2](https://www.oreilly.com/library/view/learning-spark-2nd/9781492050032/)

---
*Module 5 — Introduction to Apache Spark &nbsp;|&nbsp; Big Data Analytics*